# Ingesta de datos
En este Notebook se realizará un analisis de como vienen los datos de la fuente RAMA, además, contendrá el codigo de extracción automatizada y transformación de archivos .xls a CSV.
La documentación contiene una explicación más detallada sobre como están organizados los datos en la fuente principal RAMA.
Dado que la cantidad de datos no es tan grande, decidí que se utilizará Jupyter Notebok en Google Colab para los códigos y se conectará a Google Drive, el cúal será utilizado como repositorio de los datos crudos y procesados.


In [ ]:
#Conectamos Colab con Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Librerías
Estas son las librerias que utilizaré para la ingesta y analisis.
* Pathlib:  Para trabajar con rutas y directorios
* requests: Nos permite realizar solicitudes HTTP para descargar archivos desde la pagina de RAMA
* zipfile: Nos permitirá trabajar con archivos comprimidos en formato ZIP
* pandas: Proporciona herramientas para manipulación y análisis de datos tabulares
* numpy: Proporciona funciones para cálculos numéricos y manejo de valores


In [ ]:
from pathlib import Path
import requests
import zipfile
import pandas as pd
import numpy as np

## Configuración del entorno
Las carpetas en Google Drive se han creado de manera manual antes de realizar este proceso de ingesta y análisis.

Utilizando la libreria pathlib podremos definir las rutas que serán utilizadas durante todo el proceso de ingesta. La estructura es la siguiente:

* ProyectoRAMA
  - DATA
    - LANDING
    - RAW
    - PROCESSED
  - notebooksColab
  - Documentacion

Dentro de la documentación se detalla lo que contendrá cada una de las carpetas.

In [ ]:
BASE_PATH = Path("/content/drive/MyDrive/ProyectoRAMA") # Ruta raíz del proyecto en Google Drive
DATA_PATH = BASE_PATH / "DATA" # Ruta de la carpeta DATA
LANDING_PATH = DATA_PATH / "LANDING" # Ruta de la carpeta LANDING
RAW_PATH = DATA_PATH / "RAW" # Ruta de la carpeta RAW
PROCESSED_PATH = DATA_PATH / "PROCESSED" # Ruta de la carpeta PROCESSED

## Configuración de la fuente de datos

De la *Red Automática de Monitoreo Atmosférico* (RAMA) se obtuvo el enlace al repositorio oficial de Datos Abiertos del Sistema de Monitoreo Atmosférico (SIMAT) de la Ciudad de México, que es el subsistema encargado de medir de forma continua los principales contaminantes del aire y las variables meteorológicas en la Zona Metropolitana del Valle de México.

In [ ]:
# Usando pathlib definimos la URL base de descarga de los archivos RAMA
BASE_URL = "http://aire.cdmx.gob.mx/descargas/Opendata/Bases_publicas/RAMA/"


## Descarga y exploración de archivo de prueba
En esta parte probamos descargar un archivo zip para probar que la URL sea correcta y ver que el archivo se guarde en la carpeta correspondiente.
Usaré de ejemplo el archivo 11RAMA (En la documentación se detalla como estan compuestos los nombres de los archivos zip y su contenido).

In [ ]:
# Definimos el nombre del archivo a descargar
file_name = "11RAMA.zip"
file_url = BASE_URL + file_name # Construye la URL completa del archivo http://aire.cdmx.gob.mx/descargas/Opendata/Bases_publicas/RAMA/11RAMA.zip

output_file = LANDING_PATH / file_name # Define la ruta donde se almacenará el archivo descargado /content/drive/MyDrive/ProyectoRAMA/DATA/LANDING

print (file_url,"\n", output_file) # Muestra la URL del archivo

http://aire.cdmx.gob.mx/descargas/Opendata/Bases_publicas/RAMA/11RAMA.zip 
 /content/drive/MyDrive/ProyectoRAMA/DATA/LANDING/11RAMA.zip


In [ ]:
# Abre el archivo ZIP en modo lectura
with zipfile.ZipFile(output_file, "r") as zip_file:

    # Obtiene la lista de archivos contenidos en el ZIP
    file_list = zip_file.namelist()

for file in file_list:
    print(file) # Muestra el nombre de cada archivo

2011NOX.xls
2011O3.xls
2011PM10.xls
2011PM25.xls
2011PMCO.xls
2011SO2.xls
2011CO.xls
2011NO.xls
2011NO2.xls


**Resultado:** El archivo zip se guardo de manera correcta en la carpeta LANDING y dentro del archivo zip se encuentran los 9 archivos .xls de cada contaminante correspondiente al año 2011.

## Exploración de la estructura de un archivo XLS

Antes de continuar con la extracción completa de los años previstos(2011-2026) se analizará el contenido de uno de los contaminantes, en este caso Ozono, para confirmar la estructura de los datos, nombres de columnas, formato de fechas, etc.

In [ ]:
# Se define el nombre del archivo que se va analizar
target_file = "2011O3.xls"

# Abre el archivo ZIP en modo lectura
with zipfile.ZipFile(output_file, "r") as zip_file:

    # Extrae únicamente el archivo seleccionado a una carpeta temporal
    zip_file.extract(target_file, "/content/temp_rama")

# Se construye la ruta completa del archivo extraído
xls_file = Path("/content/temp_rama") / target_file # /content/drive/MyDrive/ProyectoRAMA/content/temp_rama/2011O3.xls

# Lee el archivo Excel
df = pd.read_excel(
    xls_file,
    engine="xlrd"
)

# Muestra las primeras 5 filas
df.head()

,FECHA,HORA,ACO,CAM,CHO,COY,CUA,FAC,IZT,LLA,...,SAG,SJA,SUR,TAH,TLA,TLI,TPN,UIZ,VIF,XAL
0,2011-01-01,1,7,-99,3,4,16,4,1,-99,...,3,-99,5,14,2,-99,33,2,-99,2
1,2011-01-01,2,4,-99,5,4,20,4,0,-99,...,3,-99,6,10,2,-99,32,0,-99,2
2,2011-01-01,3,5,-99,8,4,21,5,1,-99,...,4,-99,6,6,2,-99,26,0,-99,1
3,2011-01-01,4,5,-99,5,4,23,5,1,-99,...,4,-99,5,9,2,-99,25,0,-99,2
4,2011-01-01,5,4,-99,3,4,19,3,1,-99,...,4,-99,5,11,2,-99,25,0,-99,2


In [ ]:
# Dimensiones del dataset
print(df.shape)
print(df.columns.tolist())

(8760, 25)
['FECHA', 'HORA', 'ACO', 'CAM', 'CHO', 'COY', 'CUA', 'FAC', 'IZT', 'LLA', 'LPR', 'MER', 'MON', 'NEZ', 'PED', 'SAG', 'SJA', 'SUR', 'TAH', 'TLA', 'TLI', 'TPN', 'UIZ', 'VIF', 'XAL']


In [ ]:
# información general de las columnas
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   FECHA   8760 non-null   datetime64[ns]
 1   HORA    8760 non-null   int64         
 2   ACO     8760 non-null   int64         
 3   CAM     8760 non-null   int64         
 4   CHO     8760 non-null   int64         
 5   COY     8760 non-null   int64         
 6   CUA     8760 non-null   int64         
 7   FAC     8760 non-null   int64         
 8   IZT     8760 non-null   int64         
 9   LLA     8760 non-null   int64         
 10  LPR     8760 non-null   int64         
 11  MER     8760 non-null   int64         
 12  MON     8760 non-null   int64         
 13  NEZ     8760 non-null   int64         
 14  PED     8760 non-null   int64         
 15  SAG     8760 non-null   int64         
 16  SJA     8760 non-null   int64         
 17  SUR     8760 non-null   int64         
 18  TAH     

## Hallazgos de la estructura de datos

- El archivo contiene 8,760 registros, los cuales equivalen a una medición por hora durante todo el año 2011.
- La estructura de los datos está compuesta por dos columnas FECHA y HORA, y otras 23 columnas correspondientes a las estaciones de monitoreo.
- La columna FECHA es tipo datetime, mientras que las columnas de medición se encuentran como valores enteros.
- Cada fila representa una observación horaria y cada columna de estación contiene la concentración registrada para el contaminante analizado.
- Se usa -99 para representar datos faltantes.
- Las estaciones se encuentran representadas como columnas.

La estructura de los datos coincide con el de la documentación proporcionada por RAM.
Para facilitar el análisis, integración y almacenamiento de la información, es necesario transformar los datos a un formato en el cual cada registro represente una combinación de fecha, hora, estación, contaminante y valor medido.

## Validación de consistencia entre contaminantes
Vamos a hacer nuevamente un analisis a otro contaminante, ahora Partículas suspendidas menores a 10 micrómetros, correspondiente al mismo año que el anterior (2011) para verificar que sean consistentes.

In [ ]:
# Se define el nombre del archivo que se va analizar
target_file = "2011PM10.xls"

# Abre el archivo ZIP en modo lectura
with zipfile.ZipFile(output_file, "r") as zip_file:

    # Extrae únicamente el archivo seleccionado a una carpeta temporal
    zip_file.extract(target_file, "/content/temp_rama")

# Se construye la ruta completa del archivo extraído
xls_file = Path("/content/temp_rama") / target_file # /content/drive/MyDrive/ProyectoRAMA/content/temp_rama/2011PM10.xls

# Lee el archivo Excel
df = pd.read_excel(
    xls_file,
    engine="xlrd"
)

# Muestra las primeras 5 filas
df.head()

,FECHA,HORA,ACO,CAM,FAC,IZT,MER,PED,SAG,SUR,TAH,TLA,TLI,UIZ,VIF,XAL
0,2011-01-01,1,-99,-99,94,103,46,49,113,155,82,74,97,-99,126,67
1,2011-01-01,2,-99,-99,115,163,76,68,186,124,146,107,89,-99,425,104
2,2011-01-01,3,-99,-99,124,169,90,74,228,188,-99,122,78,-99,374,70
3,2011-01-01,4,-99,-99,90,167,108,68,228,183,-99,133,57,-99,299,101
4,2011-01-01,5,-99,-99,57,138,135,52,179,136,73,124,96,-99,340,100


In [ ]:
print(df.shape)
print(df.columns.tolist())
df.info()

(8760, 16)
['FECHA', 'HORA', 'ACO', 'CAM', 'FAC', 'IZT', 'MER', 'PED', 'SAG', 'SUR', 'TAH', 'TLA', 'TLI', 'UIZ', 'VIF', 'XAL']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 16 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   FECHA   8760 non-null   datetime64[ns]
 1   HORA    8760 non-null   int64         
 2   ACO     8760 non-null   int64         
 3   CAM     8760 non-null   int64         
 4   FAC     8760 non-null   int64         
 5   IZT     8760 non-null   int64         
 6   MER     8760 non-null   int64         
 7   PED     8760 non-null   int64         
 8   SAG     8760 non-null   int64         
 9   SUR     8760 non-null   int64         
 10  TAH     8760 non-null   int64         
 11  TLA     8760 non-null   int64         
 12  TLI     8760 non-null   int64         
 13  UIZ     8760 non-null   int64         
 14  VIF     8760 non-null   int64         
 15  XAL     8760 

**Conlusión:** Los contaminantes mantienen la misma estructura basada en columnas FECHA y HORA, y columnas de estaciones de monitoreo.
Se observó que la cantidad de estaciones monitoreadas varía entre contaminantes. Mientras que el archivo de ozono (O3) contiene mediciones para 23 estaciones, el archivo de PM10 incluye únicamente 14 estaciones.

Lo que indica que la disponibilidad de mediciones depende de la infraestructura instalada en cada estación y contaminante. Por lo tanto no habrá una cobertura uniforme de estaciones para todos los contaminantes.

## Automatizacion de descarga de los archivos históricos.
 Se van a descargar los archivos .zip correspondientes a los años 2011-2026.
 Los archivos descargados se almacenarán en la carpeta LANDING conservando su formato original (.zip), con el propósito de mantener una copia de la información.

Además de que se importará la libreria **datetime** para trabajar con fechas y horas que segun la documentación oficial "El módulo datetime proporciona clases para manipular fechas y horas. Su principal objetivo es poder extraer campos de forma eficiente para su posterior manipulación o formateo."

In [ ]:
from datetime import datetime # Para trabajar con fechas y horas
# Definimos el año inicial de descarga
start_year = 2011
# Definimos el año final de descarga
end_year = datetime.now().year #Dará el año en curso, para este momento es el año 2026

In [ ]:
# Recorre todos los años del rango definido

for year in range(start_year, end_year + 1):
    short_year = str(year)[2:] # Obtiene los dos últimos dígitos del año: 11,12,13,14,15...
    file_name = f"{short_year}RAMA.zip" # Construye el nombre del archivo ZIP: 11RAMA.zip, 12RAMA.zip,13RAMA.zip
    file_url = BASE_URL + file_name # Construye la URL completa de descarga: http://aire.cdmx.gob.mx/descargas/Opendata/Bases_publicas/RAMA/11RAMA.zip
    output_file = LANDING_PATH / file_name # Define la ruta donde se almacenará el archivo: /content/drive/MyDrive/ProyectoRAMA/DATA/LANDING/11RAMA.zip

    # Verifica si el archivo ya existe
    if output_file.exists():

        print(f"{file_name} ya existe") # Informa que el archivo ya fue descargado previamente
        continue # Continúa con el siguiente año

    try:

        # Realiza la solicitud HTTP
        response = requests.get(file_url)
        if response.status_code == 200:  # Verifica que la descarga fue exitosa

            with open(output_file, "wb") as file: # Guarda el archivo descargado
                file.write(response.content) # Escribe el contenido del archivo

            # Informa que la descarga fue exitosa
            print(f"Descargado: {file_name}")

        else:

            # Informa que el archivo no se encontró
            print(f"Archivo no disponible: {file_name}")

    except Exception as error:

        # Muestra el error ocurrido
        print(f"Error en {file_name}: {error}")

print("Proceso Finalizado")

11RAMA.zip ya existe
12RAMA.zip ya existe
13RAMA.zip ya existe
14RAMA.zip ya existe
15RAMA.zip ya existe
16RAMA.zip ya existe
17RAMA.zip ya existe
18RAMA.zip ya existe
19RAMA.zip ya existe
20RAMA.zip ya existe
21RAMA.zip ya existe
22RAMA.zip ya existe
23RAMA.zip ya existe
24RAMA.zip ya existe
25RAMA.zip ya existe
26RAMA.zip ya existe
Proceso Finalizado


**Conclusion:** Los archivos .zip se descargaron con éxito en la carpeta LANDING, se debe proceder a extraer los archivos .xls en la carpeta RAW para despues transformarlos a formato .csv

## Extracción de archivos XLS

Se extraen los archivos .xls contenidos en los archivos comprimidos descargados desde RAMA. Los archivos se almacenan en la carpeta RAW, para preservar su estructura original.
Como en los pasos anteriores se utilizará la libreria **zipfile** que segun la documentación oficial "Este módulo proporciona herramientas para crear, leer, escribir, agregar y listar un archivo ZIP"

In [ ]:
# Recorre todos los archivos ZIP disponibles
zip_files = list(LANDING_PATH.glob("*.zip"))

for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as zip_file:
        archivos_internos = zip_file.namelist() # Obtiene la lista de lo que hay dentro de este ZIP específico

        # Si alguno de estos archivos ya existe en la carpeta RAW
        # Creamos una lista de los archivos que faltan por extraer
        archivos_faltantes = [
            archivo for archivo in archivos_internos
            if not (RAW_PATH / archivo).exists()
        ]
        # Si no falta ninguno, nos saltamos el ZIP completo
        if len(archivos_faltantes) == 0:
            print(f"{zip_path.name} ya existe en RAW.")
            continue

        # Si faltan algunos, extraemos solo los que faltan
        elif len(archivos_faltantes) < len(archivos_internos):
            print(f"Extracción parcial {zip_path.name} ({len(archivos_faltantes)} archivos nuevos)")

            for archivo in archivos_faltantes: # Se extraen los archivos faltantes
                zip_file.extract(archivo, RAW_PATH)

        else:
            # Si no existe ninguno, extracción rápida de todo el bloque
            print(f"[Extracción completa: {zip_path.name}")
            zip_file.extractall(RAW_PATH)

print("Proceso Finalizado")

11RAMA.zip ya existe en RAW.
12RAMA.zip ya existe en RAW.
13RAMA.zip ya existe en RAW.
14RAMA.zip ya existe en RAW.
15RAMA.zip ya existe en RAW.
16RAMA.zip ya existe en RAW.
17RAMA.zip ya existe en RAW.
18RAMA.zip ya existe en RAW.
19RAMA.zip ya existe en RAW.
20RAMA.zip ya existe en RAW.
21RAMA.zip ya existe en RAW.
22RAMA.zip ya existe en RAW.
23RAMA.zip ya existe en RAW.
24RAMA.zip ya existe en RAW.
25RAMA.zip ya existe en RAW.
26RAMA.zip ya existe en RAW.
Proceso Finalizado


In [ ]:
# Obtiene todos los archivos XLS almacenados en RAW
xls_files = list(RAW_PATH.glob("*.xls"))

# Con len se muestra la cantidad total de archivos encontrados
print(f"Archivos .xls encontrados: {len(xls_files)}")

Archivos .xls encontrados: 144


**Conclusión:** Como resultado de la extracción se obtuvieron 144 archivos .xls, 9 archivos para cada contaminante en 1 año, correspondientes al período 2011–2026 guardados en la carpeta RAW.

Estos código se puede volver a ejecutar sin temor a que se repitan archivos.

## Verificación de Integridad de archivos
Antes de continuar con la conversión de archivos .xls a csv verificaremos que los archivos esten en buenas condiciones, que no haya archivos vacios, duplicados o corruptos.

In [ ]:
# Recorre todos los archivos XLS encontrados
for file in sorted(xls_files):

    # Muestra el nombre del archivo
    print(file.name)

2011CO.xls
2011NO.xls
2011NO2.xls
2011NOX.xls
2011O3.xls
2011PM10.xls
2011PM25.xls
2011PMCO.xls
2011SO2.xls
2012CO.xls
2012NO.xls
2012NO2.xls
2012NOX.xls
2012O3.xls
2012PM10.xls
2012PM25.xls
2012PMCO.xls
2012SO2.xls
2013CO.xls
2013NO.xls
2013NO2.xls
2013NOX.xls
2013O3.xls
2013PM10.xls
2013PM25.xls
2013PMCO.xls
2013SO2.xls
2014CO.xls
2014NO.xls
2014NO2.xls
2014NOX.xls
2014O3.xls
2014PM10.xls
2014PM25.xls
2014PMCO.xls
2014SO2.xls
2015CO.xls
2015NO.xls
2015NO2.xls
2015NOX.xls
2015O3.xls
2015PM10.xls
2015PM25.xls
2015PMCO.xls
2015SO2.xls
2016CO.xls
2016NO.xls
2016NO2.xls
2016NOX.xls
2016O3.xls
2016PM10.xls
2016PM25.xls
2016PMCO.xls
2016SO2.xls
2017CO.xls
2017NO.xls
2017NO2.xls
2017NOX.xls
2017O3.xls
2017PM10.xls
2017PM25.xls
2017PMCO.xls
2017SO2.xls
2018CO.xls
2018NO.xls
2018NO2.xls
2018NOX.xls
2018O3.xls
2018PM10.xls
2018PM25.xls
2018PMCO.xls
2018SO2.xls
2019CO.xls
2019NO.xls
2019NO2.xls
2019NOX.xls
2019O3.xls
2019PM10.xls
2019PM25.xls
2019PMCO.xls
2019SO2.xls
2020CO.xls
2020NO.xls
2020NO

In [ ]:
# Muestra el tamaño de cada archivo XLS
for file in sorted(xls_files):

    # Obtiene el tamaño en megabytes
    size_mb = file.stat().st_size / 1024 / 1024

    # Muestra el nombre y tamaño del archivo
    print(f"{file.name} - {size_mb:.2f} MB")

2011CO.xls - 1.77 MB
2011NO.xls - 1.60 MB
2011NO2.xls - 1.60 MB
2011NOX.xls - 1.60 MB
2011O3.xls - 1.55 MB
2011PM10.xls - 1.10 MB
2011PM25.xls - 0.95 MB
2011PMCO.xls - 0.74 MB
2011SO2.xls - 1.55 MB
2012CO.xls - 1.84 MB
2012NO.xls - 1.81 MB
2012NO2.xls - 1.81 MB
2012NOX.xls - 1.81 MB
2012O3.xls - 1.71 MB
2012PM10.xls - 1.20 MB
2012PM25.xls - 1.05 MB
2012PMCO.xls - 0.85 MB
2012SO2.xls - 1.76 MB
2013CO.xls - 1.81 MB
2013NO.xls - 1.80 MB
2013NO2.xls - 1.80 MB
2013NOX.xls - 1.80 MB
2013O3.xls - 1.80 MB
2013PM10.xls - 1.40 MB
2013PM25.xls - 1.05 MB
2013PMCO.xls - 0.84 MB
2013SO2.xls - 1.75 MB
2014CO.xls - 1.84 MB
2014NO.xls - 1.86 MB
2014NO2.xls - 1.86 MB
2014NOX.xls - 1.86 MB
2014O3.xls - 1.86 MB
2014PM10.xls - 1.41 MB
2014PM25.xls - 1.11 MB
2014PMCO.xls - 0.85 MB
2014SO2.xls - 1.81 MB
2015CO.xls - 2.06 MB
2015NO.xls - 2.06 MB
2015NO2.xls - 2.06 MB
2015NOX.xls - 2.06 MB
2015O3.xls - 2.16 MB
2015PM10.xls - 1.60 MB
2015PM25.xls - 1.40 MB
2015PMCO.xls - 1.05 MB
2015SO2.xls - 2.01 MB
2016CO.xls

**Conclusión:** No se encontraron archivos duplicados y el tamaño de los archivos no es inusual. Los achivos de 2026 son más pequeños que el resto debido a que el año sigue en curso y aun faltan datos.

## Conversión de archivos XLS a CSV

Se decidio cambiar el formato de .xls a .csv debido a que .csv es texto plano y python puede leer y procesar un CSV hasta 20 veces más rápido porque va directo a los datos, sin perder tiempo descifrando estilos (en la documentación se explica con más detalle).

In [ ]:
#Transformamos los .xls a csv
for xls_file in xls_files:
    csv_name = f"{xls_file.stem}.csv"  # Construye el nombre del archivo CSV (.stem obtiene el nombre sin el .xls)
    csv_path = RAW_PATH / csv_name #Se define la ruta  /content/drive/MyDrive/ProyectoRAMA/DATA/RAW

    # Si el CSV ya existe, nos saltamos todo el proceso
    if csv_path.exists():
        print(f"{csv_name} ya fue convertido previamente.")
        continue

    try:
    # Si no existe se lee el excel
        df = pd.read_excel(
            xls_file,
            engine="xlrd"
        )

    # Guardamos el DataFrame como CSV en la carpeta RAW
        df.to_csv(
            csv_path,
            index=False,
            encoding="utf-8"
        )

        print(f" Convertido: {csv_name}")

    except Exception as error:

    # En caso de que un archivo XLS esté corrupto
        print(f"Error en el archivo: {xls_file.name}: {error}")

print("Proceso Finalizado")

2011NOX.csv ya fue convertido previamente.
2011O3.csv ya fue convertido previamente.
2011PM10.csv ya fue convertido previamente.
2011PM25.csv ya fue convertido previamente.
2011PMCO.csv ya fue convertido previamente.
2011SO2.csv ya fue convertido previamente.
2011CO.csv ya fue convertido previamente.
2011NO.csv ya fue convertido previamente.
2011NO2.csv ya fue convertido previamente.
2012CO.csv ya fue convertido previamente.
2012NO.csv ya fue convertido previamente.
2012NO2.csv ya fue convertido previamente.
2012NOX.csv ya fue convertido previamente.
2012O3.csv ya fue convertido previamente.
2012PM10.csv ya fue convertido previamente.
2012PM25.csv ya fue convertido previamente.
2012PMCO.csv ya fue convertido previamente.
2012SO2.csv ya fue convertido previamente.
2013CO.csv ya fue convertido previamente.
2013NO.csv ya fue convertido previamente.
2013NO2.csv ya fue convertido previamente.
2013NOX.csv ya fue convertido previamente.
2013O3.csv ya fue convertido previamente.
2013PM10.csv y

In [ ]:
# Obtiene todos los archivos XLS almacenados en RAW
xls_files = list(RAW_PATH.glob("*.xls"))
print(f"Archivos XLS encontrados: {len(xls_files)}") # Muestra la cantidad total de archivos encontrados

# Obtiene todos los archivos CSV generados
csv_files = list(RAW_PATH.glob("*.csv"))
print(f"Archivos CSV encontrados: {len(csv_files)}") # Muestra la cantidad de archivos encontrados

Archivos XLS encontrados: 144
Archivos CSV encontrados: 144


## Conclusión de la fase de ingesta

El codigo se puede ejecutar cada vez que se desee obtener los datos más recientes sin dañar los que ya han sido guardados o que haya repetidos.
Los archivos se descargaron, almacenaron y convirtieron los archivos de la fuente RAMA de manera correcta. Por lo que ya es posible proceder a realizar un análisis profundo de estos datos y una correcta transformación.
